In [2]:
# Load Fiji
import geopandas as gpd
from odc.geo.geom import Geometry

fiji_tiled_geoms = gpd.read_file(
    "/Users/wj/Projects/csdr/csdr-cloud-spatial/debug_geoms_tiled.geojson"
)

print(fiji_tiled_geoms["geometry"][0].bounds)

fiji_tiled_geoms = [Geometry(geom) for geom in fiji_tiled_geoms["geometry"]]

geometry_4326 = fiji_tiled_geoms[0].assign_crs("EPSG:4326")
geom_geojson = geometry_4326.geojson(simplify=0)["geometry"]

(-180.0, -24.3819587198891, -176.267461612063, -12.820507703401365)


In [ ]:
# Search and load STAC
from pystac import ItemCollection
from rustac import search_sync as rustac_search_sync

dataset_url = "../cache/datasets/seagrass/0-0-1/dep_s2_seagrass.parquet"

items = rustac_search_sync(
    dataset_url,
    intersects=geom_geojson,
    datetime="2017",
)

print(f"Found {len(items)} items")

items = ItemCollection(items)

ok_items = []
naughty_items = []
for item in items:
    print(item.bbox)
    xmin, ymin, xmax, ymax = item.bbox
    if xmin == -180.0 and xmax == 180.0:
        naughty_items.append(item)
    else:
        ok_items.append(item)

print(f"Found {len(ok_items)} ok items and {len(naughty_items)} naughty items")

print("Ok items:")
for item in ok_items:
    print(item.id, item.bbox)

print("Naughty items:")
for item in naughty_items:
    print(item.id, item.bbox)

Found 12 items
[-178.30743677626327, -20.11524267928007, -177.4450541035085, -19.298529078932667]
[-179.169819449018, -16.824114445989782, -178.30743677626327, -15.991730594839629]
[-179.169819449018, -17.65281330691017, -178.30743677626327, -16.824114445989782]
[-179.169819449018, -18.477669425954502, -178.30743677626327, -17.65281330691017]
[-179.169819449018, -19.298529078932667, -178.30743677626327, -18.477669425954502]
[-179.169819449018, -20.927664878354115, -178.30743677626327, -20.11524267928007]
[-179.169819449018, -21.73565465582795, -178.30743677626327, -20.927664878354115]
[-180.0, -15.991730594839629, 180.0, -15.15582341258985]
[-180.0, -16.824114445989782, 180.0, -15.991730594839629]
[-180.0, -17.65281330691017, 180.0, -16.824114445989782]
[-180.0, -18.477669425954502, 180.0, -17.65281330691017]
[-180.0, -19.298529078932667, 180.0, -18.477669425954502]
Found 7 ok items and 5 naughty items
Ok items:
dep_s2_seagrass_068_018_2017 [-178.30743677626327, -20.11524267928007, -17

In [4]:
# The whole of column 66 in the COG grid is straddling the antimeridian.
# Devilish behaviour!
from odc.stac import load
from pyproj import Transformer

print(
    "This geom is just in the western hemisphere. These COGs in 6933 are in both hemispheres, so they have a huge dataset (x dimension) spanning the globe."
)

# Now to try and load these to get a count in 6933.
data = load(naughty_items, crs="EPSG:6933", resolution=10, chunks={})
print(f"Loaded data with shape {data.dims}")

This geom is just in the western hemisphere. These COGs in 6933 are in both hemispheres, so they have a huge dataset (x dimension) spanning the globe.
Loaded data with shape FrozenMappingWarningOnValuesAccess({'y': 50528, 'x': 3473423, 'time': 1})


In [ ]:
# Attempted solution:
# Load in 3832
# Mask to geom in 3832
# Filter to indicator value.
# Do the count myself with area conversion.
from odc.geo.xr import mask
from pyproj import Geod

indicator = "seagrass"
value_list = [1.0]

data_3832 = load(naughty_items, crs="EPSG:3832", resolution=10, chunks={})
print(f"Loaded data with shape {data_3832.dims}")

geom_3832 = geometry_4326.to_crs("EPSG:3832")

data_3832_masked = mask(data_3832, geom_3832)

data_3832_masked[indicator] = data_3832_masked[indicator].where(
    data_3832_masked[indicator].isin(value_list)
)


# # Find antimeridian x in 3832 metres
# t = Transformer.from_crs(4326, 3832, always_xy=True)
# antimeridian_x, _ = t.transform(180, 0)

# # Split at antimeridian in 3832 space
# west = data_3832.sel(x=data_3832.x[data_3832.x < antimeridian_x])
# east = data_3832.sel(x=data_3832.x[data_3832.x >= antimeridian_x])

# west_pixel_count = int(west.notnull().sum().compute())
# east_pixel_count = int(east.notnull().sum().compute())
# print(f"West pixel count: {west_pixel_count}")
# print(f"East pixel count: {east_pixel_count}")


# Calculate ellipsoidal (geodesic) area of each pixel in the masked data, then sum.

geod = Geod(ellps="WGS84")

# For a regular grid, every pixel in a row has the same area
# Just need lat/lon of pixel corners
res = 10  # metres in EPSG:3832
t_to_4326 = Transformer.from_crs(3832, 4326, always_xy=True)

# Get pixel corner lats/lons for each row (y value)
mid_x = float(data_3832_masked.x.mean())
_, lats = t_to_4326.transform(
    [mid_x] * len(data_3832_masked.y),
    data_3832_masked.y.values.astype(float),
)
_, lats_shifted = t_to_4326.transform(
    [mid_x + res] * len(data_3832_masked.y),
    (data_3832_masked.y.values + res).astype(float),
)

# Geodesic polygon area per pixel (4 corners)
lons_0, _ = t_to_4326.transform(
    [mid_x] * len(lats), data_3832_masked.y.values.astype(float)
)
lons_1, _ = t_to_4326.transform(
    [mid_x + res] * len(lats), data_3832_masked.y.values.astype(float)
)

pixel_areas = []
for i in range(len(lats)):
    # 4-corner polygon for one pixel
    area, _ = geod.polygon_area_perimeter(
        [lons_0[i], lons_1[i], lons_1[i], lons_0[i]],
        [lats[i], lats[i], lats_shifted[i], lats_shifted[i]],
    )
    pixel_areas.append(abs(area))

print(f"Pixel areas (m^2): {pixel_areas}")

Loaded data with shape FrozenMappingWarningOnValuesAccess({'y': 48000, 'x': 9600, 'time': 1})
Pixel areas (m^2): [93.20729893818498, 93.20722253248096, 93.20714611187577, 93.20706970244646, 93.20699328929186, 93.20691686682403, 93.2068404648453, 93.20676403678954, 93.20668762177229, 93.20661121048033, 93.20653479360044, 93.20645837858319, 93.20638196542859, 93.20630553551018, 93.20622912794352, 93.20615270547569, 93.2060762885958, 93.20599986985326, 93.20592344366014, 93.20584702864289, 93.20577061735094, 93.20569418929517, 93.20561776682734, 93.20554134994745, 93.20546492189169, 93.20538850873709, 93.2053120713681, 93.20523566752672, 93.20515923574567, 93.2050828114152, 93.20500638708472, 93.20492997579277, 93.20485353656113, 93.2047771140933, 93.20470068976283, 93.2046242710203, 93.20454784296453, 93.20447140187025, 93.20439499430358, 93.2043185569346, 93.20424213446677, 93.20416570454836, 93.20408928021789, 93.2040128391236, 93.203936425969, 93.20385999418795, 93.20378355309367, 93.

In [ ]:
# Here are my results from debugging earlier:
# # Expecting
# Original value for Fiji at 100m:
# 296,420,000 m²

# 10m Skipping/Ignoring bad COGs
# 286,171,300 m²


def nice_print_area(area_m2: float) -> str:
    if area_m2 >= 1e6:
        return f"{area_m2 / 1e6:.2f} km²"
    else:
        return f"{area_m2:.2f} m²"


print(
    f"Expecting about {nice_print_area(296_420_000 - 286_171_300)} for the naughty COGs"
)

Expecting about 10.25 km² for the naughty COGs


In [12]:
# Calculate the actual area of the naughty COGs seagrass by counting pixels and multiplying by pixel area.
counts_per_row = (
    data_3832_masked[indicator].notnull().sum(dim=["x", "time"]).compute().values
)
area_m2 = float(sum(count * pa for count, pa in zip(counts_per_row, pixel_areas)))
print(f"Naughty COGs seagrass area: {nice_print_area(area_m2)}")

Naughty COGs seagrass area: 11.53 km²


In [ ]:
# from csdr.utils import xarray_calculate_area_m2

# indicator = "seagrass"
# value_list = [1.0]

# total_area_m2 = xarray_calculate_area_m2(
#     data[indicator], geometry_4326, indicator=indicator, value_list=value_list
# )